# CardioAgent-Ops — Data Mining & Audio Benchmark (Colab T4)
Fused PASCAL+PhysioNet @ `dataset/train/{normal,abnormal}`. Canonical: resample **2000Hz**, window **2s (4000 samples)**, z-norm. Replacement for `MIT/ast-finetuned-audioset` (16kHz-locked): **CNN+Transformer from scratch** (`audio_engine/models.py`), same Mel 64x128 input as all 5 models.

In [ ]:
import sys
from pathlib import Path

# Colab: mount Drive (no-op locally)
try:
    from google.colab import drive
    drive.mount("/content/drive")
except ImportError:
    pass

import os
CANDIDATES = [Path(os.environ.get("DATA_DIR", "")).parent if os.environ.get("DATA_DIR") else None,
              Path("/content/drive/MyDrive/dataset/train"),
              Path("dataset/train"), Path("../dataset/train"), Path("/content/CardioAgent-Ops/dataset/train")]
DATA_DIR = next((p for p in CANDIDATES if p and (p / "normal").is_dir()), Path("dataset/train"))
REPO = next((p for p in [Path("."), Path(".."), Path("/content/drive/MyDrive/CardioAgent-Ops"),
                         Path("/content/CardioAgent-Ops")] if (p / "audio_engine").exists()), Path("."))
sys.path.insert(0, str(REPO / "audio_engine"))
print("DATA_DIR:", DATA_DIR)
print("REPO:", REPO.resolve())
assert (DATA_DIR / "normal").is_dir() and (DATA_DIR / "abnormal").is_dir(), DATA_DIR

In [ ]:
import pandas as pd

LABEL2ID = {"normal": 0, "abnormal": 1}
records = [{"path": str(p), "label": lab, "class_name": cls}
           for cls, lab in LABEL2ID.items()
           for p in (DATA_DIR / cls).rglob("*.wav") if p.is_file()]
df = pd.DataFrame(records)
print("Total WAV files:", len(df))
print(df["class_name"].value_counts())
print(f"imbalance normal/abnormal = {sum(df.label == 0) / max(sum(df.label == 1), 1):.2f}:1")
assert len(df) > 0 and df["label"].nunique() == 2
df.head()

In [ ]:
import matplotlib.pyplot as plt

df["class_name"].value_counts().reindex(["normal", "abnormal"]).plot(kind="bar")
plt.title("Fused PASCAL+PhysioNet — class distribution")
plt.ylabel("recordings"); plt.xticks(rotation=0); plt.show()

In [ ]:
# Signal audit with stdlib wave (no soundfile needed).
import contextlib, wave
import pandas as pd

rows = []
for _, r in df.sample(min(40, len(df)), random_state=42).iterrows():
    with contextlib.closing(wave.open(r["path"], "rb")) as w:
        rows.append({"class": r["class_name"], "sr": w.getframerate(),
                   "dur": w.getnframes() / w.getframerate(), "ch": w.getnchannels()})
audit = pd.DataFrame(rows)
print(audit.groupby("class")[["sr", "dur", "ch"]].describe())
assert set(audit["sr"]) == {4000}, f"expected raw 4000Hz, got {set(audit.sr)}"
# -> justifies canonical downsample 4000 -> 2000Hz in preprocess.load_wav

In [ ]:
# Harmonization demo: raw 4000Hz vs canonical 2000Hz/2s.
import contextlib, wave
import matplotlib.pyplot as plt
import numpy as np
from preprocess import N, SR, load_wav

demo = df.sample(2, random_state=0).iloc[::-1]  # 1 per class approx
fig, ax = plt.subplots(2, 2, figsize=(12, 5))
for i, (_, r) in enumerate(demo.iterrows()):
    with contextlib.closing(wave.open(r["path"], "rb")) as w:
        raw = np.frombuffer(w.readframes(w.getnframes()), dtype=np.int16).astype(float)
        raw = raw / 32768.0
        fs = w.getframerate()
    y = load_wav(r["path"])
    assert y.shape == (N,)
    ax[i][0].plot(raw[:fs * 2]); ax[i][0].set_title(f"raw {r["class_name"]} {fs}Hz")
    ax[i][1].plot(y); ax[i][1].set_title(f"canonical {SR}Hz/2s z-norm")
plt.tight_layout(); plt.show()

In [ ]:
# Feature mining: Mel 64x128 + MFCC-mean (numpy/scipy, no librosa).
import matplotlib.pyplot as plt
from features import TIME_FRAMES, N_MELS, mel_spectrogram, mfcc_mean
from preprocess import load_wav

fig, ax = plt.subplots(2, 2, figsize=(12, 6))
for i, cls in enumerate(["normal", "abnormal"]):
    p = df.query("class_name == @cls").iloc[0]["path"]
    y = load_wav(p)
    m, c = mel_spectrogram(y), mfcc_mean(y)
    assert m.shape == (N_MELS, TIME_FRAMES) == (64, 128), m.shape
    ax[i][0].imshow(m, aspect="auto", origin="lower"); ax[i][0].set_title(f"{cls} log-Mel 64x128")
    ax[i][1].bar(range(len(c)), c); ax[i][1].set_title(f"{cls} MFCC-mean 40")
plt.tight_layout(); plt.show()

In [ ]:
# Quality audit: corrupt/flat/clipped/short counts (cleaning proof).
from quality import audit
audit(DATA_DIR)  # 0 rejects expected; short -> padded, clipped -> kept


In [ ]:
# SINGLE stratified split 80/10/10 (replaces old double-split Cells 6+9).
from sklearn.model_selection import train_test_split

train_df, temp_df = train_test_split(df, test_size=0.20, stratify=df["label"], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df["label"], random_state=42)
for n, d in [("train", train_df), ("val", val_df), ("test", test_df)]:
    print(n, len(d), dict(d["label"].value_counts().sort_index()))
train_df.to_csv("splits_train.csv", index=False)  # provenance for report

In [ ]:
# Sanity: CNN+Transformer from scratch (replaces AST, strict 2000Hz/2s).
import torch
from models import get

dev = torch.device("cuda" if torch.cuda.is_available() else "cpu")
for name in ("cnn", "cnn_transformer"):
    m = get(name).to(dev).eval()
    with torch.no_grad():
        out = m(torch.randn(2, 1, 64, 128, device=dev))
    n = sum(p.numel() for p in m.parameters())
    assert out.shape == (2, 2), (name, out.shape)
    print(f"ok {name}: {n} params, device={dev}")

In [ ]:
# Training: recall(patho)-selected, MLflow-guarded, ONNX export.
# Dogfoods audio_engine/{train,models} so notebook == repo code.
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import recall_score
from train import SpecDS
from models import get

MODEL, EPOCHS, BATCH, LR = "cnn_transformer", 3, 16, 1e-3
to_items = lambda d: list(zip(d["path"], d["label"]))
tr = DataLoader(SpecDS(to_items(train_df)), batch_size=BATCH, shuffle=True)
va = DataLoader(SpecDS(to_items(val_df)), batch_size=BATCH)
te = DataLoader(SpecDS(to_items(test_df)), batch_size=BATCH)
model, opt, lossf = get(MODEL).to(dev), None, torch.nn.CrossEntropyLoss()
opt = torch.optim.AdamW(model.parameters(), lr=LR)
try:
    import mlflow
    mlflow.set_experiment("cardio-audio")
    mlflow.start_run(run_name=f"notebook-{MODEL}")
    mlflow.log_params({"model": MODEL, "sr": 2000, "window": "2s", "epochs": EPOCHS})
    use_mlflow = True
except Exception as e:
    print("mlflow off:", e); use_mlflow = False
best = -1.0
for ep in range(EPOCHS):
    model.train(); tot = 0.0
    for x, yb in tr:
        x, yb = x.to(dev), yb.to(dev)
        opt.zero_grad(); loss = lossf(model(x), yb)
        loss.backward(); opt.step(); tot += loss.item()
    model.eval(); P, L = [], []
    with torch.no_grad():
        for x, yb in va:
            P += model(x.to(dev)).argmax(1).cpu().tolist(); L += yb.tolist()
    r = recall_score(L, P, pos_label=1, zero_division=0)
    print(f"epoch {ep+1}/{EPOCHS} loss={tot/max(len(tr),1):.4f} recall_abnormal={r:.4f}")
    if use_mlflow: mlflow.log_metrics({"loss": tot/max(len(tr),1), "recall_abnormal": r}, step=ep)
    if r > best:
        best = r; torch.save(model.state_dict(), "cardio_model.pt")
print(f"best recall_abnormal={best:.4f} -> cardio_model.pt")
if use_mlflow: mlflow.log_metric("best_recall_abnormal", best); mlflow.end_run()
try:
    m = get(MODEL); m.load_state_dict(torch.load("cardio_model.pt", map_location="cpu")); m.eval()
    torch.onnx.export(m, torch.randn(1, 1, 64, 128), "cardio_model.onnx",
                      input_names=["mel"], output_names=["logits"], dynamo=False)
    print("onnx -> cardio_model.onnx")
except Exception as e:
    print("onnx skipped:", e)

In [ ]:
# Eval: report + confusion matrix (recall on abnormal is the selection metric).
import torch
from torch.utils.data import DataLoader
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
from train import SpecDS
from models import get

m = get(MODEL); m.load_state_dict(torch.load("cardio_model.pt", map_location="cpu"))
m = m.to(dev).eval()
P, L = [], []
te = DataLoader(SpecDS(list(zip(test_df["path"], test_df["label"]))), batch_size=BATCH)
with torch.no_grad():
    for x, yb in te:
        P += m(x.to(dev)).argmax(1).cpu().tolist(); L += yb.tolist()
print(classification_report(L, P, labels=[0, 1], target_names=["normal", "abnormal"], zero_division=0))
ConfusionMatrixDisplay(confusion_matrix(L, P, labels=[0, 1]),
                       display_labels=["normal", "abnormal"]).plot()
plt.show()